# PMIL on Colab — fully self-contained

Everything (model code, training scripts, chunked WSI loading) is embedded directly in this notebook's cells — nothing is unzipped or cloned from GitHub. Running the `%%writefile` cells below recreates the modified repo on disk inside the Colab runtime, then the pipeline runs from there.

This is the same code delivered earlier as `PMIL-colab.zip`, based on the original PMIL repo (Yu et al.) with the Colab/storage changes described in `COLAB_CHANGES.md` (on-demand WSI patch reading instead of a pre-tiled ~700GB dataset, lazy feature loading, GPU auto-detect, and a couple of pre-existing upstream bugs fixed).

**You still need to supply your own WSI files and `lib/*.ckpt` files** — those are dataset-specific and can't be embedded in a generic notebook. Fill in the config cell (Cell 3) before running the rest.

Run top to bottom, or Runtime → Run all after editing the config cell.

## 1. Install dependencies (OpenSlide + openslide-python)

In [ ]:
!apt-get -qq update
!apt-get -qq install -y openslide-tools libopenslide0 > /dev/null
!pip install -q openslide-python


## 2. Recreate the repo on disk
Each cell below writes one source file. Run them in order once per fresh runtime.

In [ ]:
!mkdir -p /content/PMIL/model /content/PMIL/lib /content/PMIL/feat /content/PMIL/cluster
%cd /content/PMIL

### `model/model_maxmil.py` — ResNet34 patch encoder (unmodified from upstream)

In [ ]:
%%writefile model/model_maxmil.py
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.nn.init import xavier_uniform_
from torch.hub import load_state_dict_from_url
import numpy as np
from torchvision.models import resnet50, resnet18, alexnet

model_path = {
    "resnet50": "./states/resnet50-19c8e357.pth"
}

device = ("cuda:0" if torch.cuda.is_available() else "cpu")


class BaseLineInstanceClassifier(nn.Module):
    def __init__(self, num_classes=2):
        super(BaseLineInstanceClassifier, self).__init__()
        #self.net = resnet50(pretrained=False)
        #self.net.load_state_dict(torch.load(model_path["resnet50"]))
        self.net = resnet18(pretrained=True)
        # for param in self.net.parameters():
        #     param.required_grad = False
        # self.net.fc = nn.Linear(self.net.fc.in_features, 2)
        self.classifier = nn.Sequential(nn.Linear(1000, num_classes))

    def forward(self, x):
        x = self.net(x)
        x = self.classifier(x)
        return x

class MILInstanceClassifier(nn.Module):
    def __init__(self, num_classes=2):
        super(MILInstanceClassifier, self).__init__()
        self.net = resnet50(pretrained=False)
        self.net.load_state_dict(torch.load(model_path["resnet50"]))
        # self.net = resnet18(pretrained=True)
        self.classifier = nn.Sequential(nn.Linear(1000, num_classes))

    def forward(self, x):
        x = self.net(x)
        x = self.classifier(x)
        return x

class feature_extractor(nn.Module):
    def __init__(self):
        super(feature_extractor, self).__init__()
        self.L = 500
        self.D = 128
        self.K = 1

        self.feature_extractor_part1 = resnet18(pretrained=False)
        # for p in self.feature_extractor_part1.parameters():
        #     p.required_grads = False
        self.feature_extractor_part2 = nn.Sequential(
            # nn.Linear(50 * 4 * 4, self.L),
            nn.Linear(1000, self.L),
            nn.ReLU(),
        )
        self.feature_extractor_part1.load_state_dict(torch.load("./states/feature_extractor.pth"))

    def forward(self, x):
        # x = x.squeeze(0)
        # x = x.cuda()
        H = self.feature_extractor_part1(x)
        H = self.feature_extractor_part2(H)  # NxL
        return H

class Attention(nn.Module):
    def __init__(self):
        super(Attention, self).__init__()
        self.L = 500
        self.D = 128
        self.K = 1
        self.attention = nn.Sequential(
            nn.Linear(self.L, self.D),
            nn.Tanh(),
            nn.Linear(self.D, self.K)
        )

        self.classifier = nn.Sequential(
            nn.Linear(self.L*self.K, 1),
            nn.Sigmoid()
        )

        for m in self.modules():
            if isinstance(m, nn.Linear):
                m.weight.data = xavier_uniform_(m.weight.data)

    def forward(self, H):
        # x = x.squeeze(0)
        # x = x.cuda()
        # H = self.feature_extractor_part1(x)
        # H = self.feature_extractor_part2(H)  # NxL

        A = self.attention(H)  # NxK
        A = torch.transpose(A, 1, 0)  # KxN
        A = F.softmax(A, dim=1)  # softmax over N
        # A is attenstion weight

        M = torch.mm(A, H)  # KxL

        Y_prob = self.classifier(M)
        Y_hat = torch.ge(Y_prob, 0.5).float()  # whether Y_prob element is larger than 0.5
        # print("A size: ", A.size())
        # print("Y_prob size: ", Y_prob.size())
        return Y_prob, Y_hat, A

    # AUXILIARY METHODS
    def calculate_classification_error(self, X, Y):
        Y = Y.float()
        Y_prob, Y_hat, _ = self.forward(X)
        error = 1. - Y_hat.eq(Y).cpu().float().mean().item()
        correct = Y_hat.eq(Y).cpu().float().sum().item()
        # print("Prob: {} | Target: {} | Pred: {} ".format(Y_prob.item(), Y.item(), Y_hat.item()))
        return correct, Y_hat

    def calculate_objective(self, X, Y):
        Y = Y.float()
        Y_prob, _, A = self.forward(X)
        Y_prob = torch.clamp(Y_prob, min=1e-5, max=1. - 1e-5)
        neg_log_likelihood = -1. * (Y * torch.log(Y_prob) + (1. - Y) * torch.log(1. - Y_prob))  # negative log bernoulli

        return neg_log_likelihood, A


if __name__ == '__main__':
    net = BaseLineInstanceClassifier().cuda()
    x = torch.randn(800, 3, 1024, 1024)
    y = torch.empty(1, dtype=torch.long).random_(5)

    print(device)
    device_ids = [0, 1]
    net = nn.DataParallel(net, device_ids=device_ids)
    net.to(device)
    # x = x.to(device)
    batch_size = 64
    batch_nums = int(x.size(0)/batch_size)
    j = 0
    net.train()
    while j < batch_nums:
        batch = x[j * batch_size:(j+1) * batch_size]
        batch = batch.to(device)
        with torch.no_grad():
            score = net(batch)
        if j == 0:
            scores = score
        else:
            scores = torch.cat((scores, score), dim=0)
        j += 1
    if j*batch_size < x.size(0):
        batch = x[j * batch_size:]
        batch = batch.to(device)
        with torch.no_grad():
            score = net(batch)
        scores = torch.cat((scores, score), dim=0)
    print(scores.size())
    scores = F.softmax(scores, dim=1)
    max_score_of_patch, _ = torch.max(scores, dim=1)
    _, index = torch.max(max_score_of_patch, dim=0)
    # _, index = torch.topk(score[:, int(target)], dim=0)
    instance = x[index].unsqueeze(0)
    y = y.to(device)
    output = net(instance)
    loss = F.cross_entropy(output, y)
    print("output size: ", output.size())
    # loss = F.cross_entropy(output, y)
    #
    # loss.backward()
    print("Finish.")


__all__ = ['ResNet','resnet34']


model_urls = {
    'resnet18': 'https://download.pytorch.org/models/resnet18-5c106cde.pth',
    'resnet34': 'https://download.pytorch.org/models/resnet34-333f7ec4.pth',
}


def conv3x3(in_planes, out_planes, stride=1, groups=1, dilation=1):
    """3x3 convolution with padding"""
    return nn.Conv2d(in_planes, out_planes, kernel_size=3, stride=stride,
                     padding=dilation, groups=groups, bias=False, dilation=dilation)


def conv1x1(in_planes, out_planes, stride=1):
    """1x1 convolution"""
    return nn.Conv2d(in_planes, out_planes, kernel_size=1, stride=stride, bias=False)


class BasicBlock(nn.Module):
    expansion = 1

    def __init__(self, inplanes, planes, stride=1, downsample=None, groups=1,
                 base_width=64, dilation=1, norm_layer=None):
        super(BasicBlock, self).__init__()
        if norm_layer is None:
            norm_layer = nn.BatchNorm2d
        if groups != 1 or base_width != 64:
            raise ValueError('BasicBlock only supports groups=1 and base_width=64')
        if dilation > 1:
            raise NotImplementedError("Dilation > 1 not supported in BasicBlock")
        # Both self.conv1 and self.downsample layers downsample the input when stride != 1
        self.conv1 = conv3x3(inplanes, planes, stride)
        self.bn1 = norm_layer(planes)
        self.relu = nn.ReLU(inplace=True)
        self.conv2 = conv3x3(planes, planes)
        self.bn2 = norm_layer(planes)
        self.downsample = downsample
        self.stride = stride

    def forward(self, x):
        identity = x

        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)

        out = self.conv2(out)
        out = self.bn2(out)

        if self.downsample is not None:
            identity = self.downsample(x)

        out += identity
        out = self.relu(out)

        return out


class Bottleneck(nn.Module):
    expansion = 4

    def __init__(self, inplanes, planes, stride=1, downsample=None, groups=1,
                 base_width=64, dilation=1, norm_layer=None):
        super(Bottleneck, self).__init__()
        if norm_layer is None:
            norm_layer = nn.BatchNorm2d
        width = int(planes * (base_width / 64.)) * groups
        # Both self.conv2 and self.downsample layers downsample the input when stride != 1
        self.conv1 = conv1x1(inplanes, width)
        self.bn1 = norm_layer(width)
        self.conv2 = conv3x3(width, width, stride, groups, dilation)
        self.bn2 = norm_layer(width)
        self.conv3 = conv1x1(width, planes * self.expansion)
        self.bn3 = norm_layer(planes * self.expansion)
        self.relu = nn.ReLU(inplace=True)
        self.downsample = downsample
        self.stride = stride

    def forward(self, x):
        identity = x

        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)

        out = self.conv2(out)
        out = self.bn2(out)
        out = self.relu(out)

        out = self.conv3(out)
        out = self.bn3(out)

        if self.downsample is not None:
            identity = self.downsample(x)

        out += identity
        out = self.relu(out)

        return out


class ResNet(nn.Module):

    def __init__(self, block, layers, num_classes=1000, zero_init_residual=False,
                 groups=1, width_per_group=64, replace_stride_with_dilation=None,
                 norm_layer=None):
        super(ResNet, self).__init__()
        if norm_layer is None:
            norm_layer = nn.BatchNorm2d
        self._norm_layer = norm_layer

        self.inplanes = 64
        self.dilation = 1
        if replace_stride_with_dilation is None:
            # each element in the tuple indicates if we should replace
            # the 2x2 stride with a dilated convolution instead
            replace_stride_with_dilation = [False, False, False]
        if len(replace_stride_with_dilation) != 3:
            raise ValueError("replace_stride_with_dilation should be None "
                             "or a 3-element tuple, got {}".format(replace_stride_with_dilation))
        self.groups = groups
        self.base_width = width_per_group
        self.conv1 = nn.Conv2d(3, self.inplanes, kernel_size=7, stride=2, padding=3,
                               bias=False)
        self.bn1 = norm_layer(self.inplanes)
        self.relu = nn.ReLU(inplace=True)
        self.maxpool = nn.MaxPool2d(kernel_size=3, stride=2, padding=1)
        self.layer1 = self._make_layer(block, 64, layers[0])
        self.layer2 = self._make_layer(block, 128, layers[1], stride=2,
                                       dilate=replace_stride_with_dilation[0])
        self.layer3 = self._make_layer(block, 256, layers[2], stride=2,
                                       dilate=replace_stride_with_dilation[1])
        self.layer4 = self._make_layer(block, 512, layers[3], stride=2,
                                       dilate=replace_stride_with_dilation[2])
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(512 * block.expansion, num_classes)

        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            elif isinstance(m, (nn.BatchNorm2d, nn.GroupNorm)):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)

        # Zero-initialize the last BN in each residual branch,
        # so that the residual branch starts with zeros, and each residual block behaves like an identity.
        # This improves the model by 0.2~0.3% according to https://arxiv.org/abs/1706.02677
        if zero_init_residual:
            for m in self.modules():
                if isinstance(m, Bottleneck):
                    nn.init.constant_(m.bn3.weight, 0)
                elif isinstance(m, BasicBlock):
                    nn.init.constant_(m.bn2.weight, 0)

    def _make_layer(self, block, planes, blocks, stride=1, dilate=False):
        norm_layer = self._norm_layer
        downsample = None
        previous_dilation = self.dilation
        if dilate:
            self.dilation *= stride
            stride = 1
        if stride != 1 or self.inplanes != planes * block.expansion:
            downsample = nn.Sequential(
                conv1x1(self.inplanes, planes * block.expansion, stride),
                norm_layer(planes * block.expansion),
            )

        layers = []
        layers.append(block(self.inplanes, planes, stride, downsample, self.groups,
                            self.base_width, previous_dilation, norm_layer))
        self.inplanes = planes * block.expansion
        for _ in range(1, blocks):
            layers.append(block(self.inplanes, planes, groups=self.groups,
                                base_width=self.base_width, dilation=self.dilation,
                                norm_layer=norm_layer))

        return nn.Sequential(*layers)

    def forward(self, x):
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu(x)
        x = self.maxpool(x)

        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)

        x = self.avgpool(x)
        x = x.reshape(x.size(0), -1)
        # x = self.fc(x)

        return self.fc(x), x


def resnet(arch, block, layers, pretrained, progress, **kwargs):
    model = ResNet(block, layers, **kwargs)
    if pretrained:
        state_dict = load_state_dict_from_url(model_urls[arch],
                                              progress=progress)

        model.load_state_dict(state_dict, strict=False)
    return model

def resnet18(pretrained=False, progress=True, **kwargs):
    r"""ResNet-18 model from
    `"Deep Residual Learning for Image Recognition" <https://arxiv.org/pdf/1512.03385.pdf>`_

    Args:
        pretrained (bool): If True, returns a model pre-trained on ImageNet
        progress (bool): If True, displays a progress bar of the download to stderr
    """
    return resnet('resnet18', BasicBlock, [2, 2, 2, 2], pretrained, progress,
                   **kwargs)

def resnet34(pretrained=False, progress=True, **kwargs):
    """Constructs a ResNet-34 model.

    Args:
        pretrained (bool): If True, returns a model pre-trained on ImageNet
        progress (bool): If True, displays a progress bar of the download to stderr
    """
    return resnet('resnet34', BasicBlock, [3, 4, 6, 3], pretrained, progress,
                   **kwargs)



### `model/__init__.py` — new: re-exports `resnet34` (missing upstream; `model_pmil.py`'s `from model import resnet34` couldn't otherwise resolve)

In [ ]:
%%writefile model/__init__.py
from model.model_maxmil import resnet34


### `model/model_pmil.py` — prototype-based MIL classifier (unmodified from upstream)

In [ ]:
%%writefile model/model_pmil.py
import torch.nn as nn
import torch.nn.functional as F
import torch
from model import resnet34


class IClassifier(nn.Module):
    def __init__(self, mil_model, feature_size=512, out_class=2):
        super(IClassifier, self).__init__()
        temp = resnet34(True)
        temp.fc = nn.Linear(temp.fc.in_features, 2)
        ch = torch.load(mil_model, map_location='cpu', weights_only=False)
        temp.load_state_dict(ch['state_dict'])
        self.features = nn.Sequential(*list(temp.children())[:-1])
        self.fc = temp.fc

    def forward(self, features):
        x = self.fc(features)
        return features, x


class BClassifier(nn.Module):
    def __init__(self, num_cluster, feature_size=512, input_size=128, output_class=2, dropout=0):
        super(BClassifier, self).__init__()
        self.transformer = nn.Linear(feature_size, input_size)

        self.classifier = nn.Sequential(
            nn.Linear(num_cluster, 128),
            nn.ReLU(True),
            nn.Dropout(dropout),
            nn.Linear(128, output_class)
        )


    def _initialize_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight)
                nn.init.constant_(m.bias, 0)

    def forward(self, features, probs, prototype):
        test, m_indices = torch.sort(probs, 0,
                                     descending=True)
        test_array = test.detach().cpu().numpy()
        m_features = torch.index_select(features, dim=0,
                                        index=m_indices[:5, 1])

        ## Metric learning
        f = self.transformer(m_features)
        p = self.transformer(prototype)

        ## Euclidean
        similarity = Euclidean_Similarity(f, p)
        cmax, _ = torch.max(similarity, dim=1, keepdim=True)
        similarity = similarity / cmax

        ## SLN Cosine
        # Cosine
        #similarity = torch.mm(f, p.transpose(0,1))
        #similarity = similarity / torch.norm(f, p=2, dim=1, keepdim=True) / torch.norm(p, p=2)

        coding = torch.mean(similarity, dim=0, keepdim=True)
        x = self.classifier(coding)
        return x, coding


class PBMIL(nn.Module):
    def __init__(self, i_classifier, b_classifier):
        super(PBMIL, self).__init__()
        self.i_classifier = i_classifier
        self.b_classifier = b_classifier

    def forward(self, x, prototype):
        features, score = self.i_classifier(x)
        probs = F.softmax(score, dim=1)
        output, codings = self.b_classifier(features, probs, prototype)
        return score, output, codings


def Euclidean_Similarity(tensor_a, tensor_b):
    device = tensor_a.device
    output = torch.zeros(tensor_a.size(0), tensor_b.size(0), device=device)
    for i in range(tensor_a.size(0)):
        output[i, :] = torch.pairwise_distance(tensor_a[i, :], tensor_b)
    return output


### `Early_Stopping.py` — checkpointing/early stop (fixed: `np.Inf` → `np.inf` for NumPy 2.x, and safe with or without `DataParallel`)

In [ ]:
%%writefile Early_Stopping.py
"""
Early stopping object

from https://github.com/Bjarten/early-stopping-pytorch

"""

import numpy as np
import torch

class EarlyStopping:
    """Early stops the training if validation loss doesn't improve after a given patience."""
    def __init__(self, model_path, patience=7, verbose=False):
        """
        Args:
            patience (int): How long to wait after last time validation loss improved.
                            Default: 7
            verbose (bool): If True, prints a message for each validation loss improvement.
                            Default: False
        """
        self.patience = patience
        self.verbose = verbose
        self.counter = 0
        self.best_score = None
        self.early_stop = False
        self.val_loss_min = np.inf
        self.model_path = model_path

    def __call__(self, val_loss, best_acc, model):

        score = -val_loss

        if self.best_score is None:
            self.best_score = score
            self.save_checkpoint(val_loss, best_acc, model)
        elif score < self.best_score:
            self.counter += 1
            print('EarlyStopping counter: {} out of {}'.format(self.counter, self.patience))
            if self.counter >= self.patience:
                self.early_stop = True
        else:
            self.best_score = score
            self.save_checkpoint(val_loss, best_acc, model)
            self.counter = 0

    def save_checkpoint(self, val_loss, best_acc, model):
        '''Saves model when validation loss decrease.'''
        if self.verbose:
            print('Validation loss decreased ({} --> {}).  Saving model ...{}'.format(self.val_loss_min, val_loss, self.model_path))
        underlying_model = model.module if hasattr(model, 'module') else model
        obj = {
            'state_dict': underlying_model.state_dict(),
            'best_acc': best_acc,
        }
        torch.save(obj, self.model_path)
        self.val_loss_min = val_loss


### `slide_cache.py` — new: on-demand, storage-bounded WSI patch reading (this is the actual chunking mechanism)

In [ ]:
%%writefile slide_cache.py
"""
On-demand, storage-bounded WSI patch access.

Replaces the assumption that every slide's patches are already pre-tiled as
.jpg files on disk (the full dataset, ~700GB for Camelyon16). Instead, a
patch is read directly from its source WSI, using the exact (x, y) coordinate
and pyramid level already recorded in lib['grid'] / lib['level'] -- the same
coordinates that would have been used to pre-tile the .jpg in the first
place. Slides are opened lazily and evicted (LRU) once more than
MAX_RESIDENT_SLIDES are open, so at most a handful of raw slides occupy disk
at any time instead of the whole dataset.

Note: because this reads the region straight from the WSI instead of via a
pre-saved .jpg, the pixels are uncompressed (no JPEG artifacts). This is a
strictly higher-fidelity version of the same patch, not a different one --
but it is not byte-identical to a .jpg round trip.

If a slide path isn't available locally (e.g. not yet downloaded from your
storage backend), wire `fetch_remote_slide` below to however your raw WSIs
are hosted (Drive, GigaDB, an internal bucket, etc). Until then it raises a
clear error rather than silently failing.
"""

import os
import shutil
import tempfile
from collections import OrderedDict

import openslide

MAX_RESIDENT_SLIDES = int(os.environ.get('PMIL_MAX_RESIDENT_SLIDES', 4))
_SCRATCH_DIR = os.environ.get('PMIL_SCRATCH_DIR', os.path.join(tempfile.gettempdir(), 'pmil_slide_cache'))
os.makedirs(_SCRATCH_DIR, exist_ok=True)


def fetch_remote_slide(slide_path: str) -> str:
    """
    Ensure `slide_path` exists locally and return a local path to it.

    Default behaviour: if the path already exists (e.g. a mounted Google
    Drive folder, or a local/Colab-uploaded copy), just use it as-is -- this
    is the common Colab setup (mount Drive, point train_lib/slide paths at
    it) and needs no further wiring.

    If you instead host raw WSIs somewhere that isn't a plain filesystem
    path (a Drive file-id map, a URL list, etc), download the single file
    into _SCRATCH_DIR here and return that local path.
    """
    if os.path.exists(slide_path):
        return slide_path

    raise FileNotFoundError(
        f"Slide not found locally: {slide_path}\n"
        "Either mount/point this path at your slide storage (e.g. Google "
        "Drive), or fill in fetch_remote_slide() in slide_cache.py with "
        "your download logic (Drive file id, URL, etc)."
    )


class SlideCache:
    """LRU cache of open openslide.OpenSlide handles, bounded to
    MAX_RESIDENT_SLIDES slides resident at once."""

    def __init__(self, max_resident=MAX_RESIDENT_SLIDES):
        self.max_resident = max_resident
        self._handles = OrderedDict()          # slide_path -> OpenSlide
        self._downloaded_copies = {}            # slide_path -> scratch copy path (if any)

    def _open(self, slide_path):
        if slide_path in self._handles:
            self._handles.move_to_end(slide_path)
            return self._handles[slide_path]

        local_path = fetch_remote_slide(slide_path)
        if local_path != slide_path:
            self._downloaded_copies[slide_path] = local_path

        handle = openslide.OpenSlide(local_path)
        self._handles[slide_path] = handle
        self._handles.move_to_end(slide_path)
        self._evict_if_needed()
        return handle

    def _evict_if_needed(self):
        while len(self._handles) > self.max_resident:
            oldest_path, oldest_handle = self._handles.popitem(last=False)
            oldest_handle.close()
            scratch_copy = self._downloaded_copies.pop(oldest_path, None)
            if scratch_copy and os.path.exists(scratch_copy):
                try:
                    os.remove(scratch_copy)
                except IsADirectoryError:
                    shutil.rmtree(scratch_copy, ignore_errors=True)

    def get_patch(self, slide_path, coord, level=0, size=256, mult=1):
        """Read exactly the patch that would have been pre-tiled at
        (coord, level, size), directly from the source WSI."""
        handle = self._open(slide_path)
        x, y = coord
        if mult != 1:
            size = int(round(size * mult))
        region = handle.read_region((x, y), level, (size, size)).convert('RGB')
        if mult != 1:
            region = region.resize((256, 256))
        return region

    def close_all(self):
        for handle in self._handles.values():
            handle.close()
        self._handles.clear()
        for scratch_copy in self._downloaded_copies.values():
            if os.path.exists(scratch_copy):
                try:
                    os.remove(scratch_copy)
                except IsADirectoryError:
                    shutil.rmtree(scratch_copy, ignore_errors=True)
        self._downloaded_copies.clear()


# Module-level singleton shared by Inferencedataset / Traindataset so all
# slide access across both goes through the same bounded cache.
#
# Note on DataLoader workers (num_workers > 0): on Linux/Colab, workers are
# started by forking. This is safe here because handles are only ever
# opened lazily inside __getitem__, which DataLoader always executes inside
# the worker processes -- never in the main process -- so nothing is open
# yet at fork time to be unsafely duplicated. Each worker ends up with its
# own independent cache, bounded to MAX_RESIDENT_SLIDES per worker.
slide_cache = SlideCache()


### `max-mil.py` — patch-level encoder training (chunked WSI access, GPU auto-detect, upstream bugs fixed)

In [ ]:
%%writefile max-mil.py
import sys
import os
import numpy as np
import argparse
import random
import openslide
import pandas as pd
import PIL.Image as Image
import torch
import torch.nn as nn
import torch.optim as optim
import torch.backends.cudnn as cudnn
import torch.nn.functional as F
import torch.utils.data as data
import torchvision.transforms as transforms
from torch.optim.lr_scheduler import MultiStepLR
import torchvision.models as models
from sklearn.metrics import roc_curve, auc
from model.model_maxmil import resnet34
from collections import OrderedDict
import time
from Early_Stopping import EarlyStopping
from slide_cache import slide_cache


np.random.seed(24)
torch.manual_seed(24)
torch.cuda.manual_seed(24)
torch.cuda.manual_seed_all(24)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False


parser = argparse.ArgumentParser(description='the standard instance-space Max-MIL')
parser.add_argument('--train_lib', type=str, default='lib/train.ckpt', help='lib to save wsi id of train set')
parser.add_argument('--val_lib', type=str, default='lib/val.ckpt', help='lib to save wsi id of valid set')
parser.add_argument('--test_lib', type=str, default='lib/test.ckpt', help='lib to save wsi id of test set')
parser.add_argument('--output', type=str, default='result', help='output directory')
parser.add_argument('--feat_dir', type=str, default='', help='path to save features')
parser.add_argument('--batch_size', type=int, default=256, help='mini-batch size')
parser.add_argument('--nepochs', type=int, default=30, help='number of epochs')
parser.add_argument('--workers', default=4, type=int, help='number of data loading workers')
parser.add_argument('--test_every', default=5, type=int, help='test on val every')
parser.add_argument('--weights', default=0.5, type=float, help='unbalanced positive class weight (default: 0.5, balanced classes)')
parser.add_argument('--k', default=1, type=int, help='top k tiles are assumed to be of the same class as the slide (default: 1, standard MIL)')
parser.add_argument('--mil_model', type=str, default='model/mil.pth', help='path to pretrained model')
parser.add_argument('--n', default=1000, type=int, help='select top n tiles')
parser.add_argument('--lr', type=float, default=1e-4, help='learning rate')
parser.add_argument('--patch_dir', type=str, default='',
                     help='unused: patches are now read on-demand from the WSI paths in --train_lib/--val_lib/--test_lib (see slide_cache.py), kept only for backward-compatible CLI compatibility')
parser.add_argument('--save_model', default=False, action='store_true')
parser.add_argument('--save_feat', default=False, action='store_true')
parser.add_argument('--feat_format', type=str, choices = ['.csv', '.npy', '.pt'], default='.csv')
parser.add_argument('--load_model', default=False, action='store_true')
parser.add_argument('--is_test', default=False, action='store_true')
parser.add_argument('--save_index', default=False, action='store_true')
parser.add_argument('--device', type=int, default=0)
parser.add_argument('--device_ids', type=int, nargs='+', default=None,
                     help='GPU ids for DataParallel; default auto-detects what is actually available (Colab typically has 1 GPU)')


global args, best_acc
args = parser.parse_args()
if torch.cuda.is_available():
    torch.cuda.set_device(args.device)
    device = torch.device(f'cuda:{args.device}')
else:
    device = torch.device('cpu')
# Colab (and most non-multi-GPU setups) only exposes 1 GPU, unlike the
# original 4-GPU default; auto-detect instead of assuming a fixed cluster.
if args.device_ids is None:
    args.device_ids = list(range(torch.cuda.device_count())) or [0]

def main():
    best_acc = 0
    os.makedirs(args.output, exist_ok=True)
    os.makedirs(args.feat_dir, exist_ok=True)
    mil_model_dir = os.path.dirname(args.mil_model)
    if mil_model_dir:
        os.makedirs(mil_model_dir, exist_ok=True)
    model = resnet34(pretrained=True)
    model.fc = nn.Linear(model.fc.in_features, 2)
    if args.load_model:
        ch = torch.load(args.mil_model, map_location='cpu', weights_only=False)
        model.load_state_dict(ch["state_dict"], strict=False)
    model.to(device)

    if torch.cuda.is_available() and len(args.device_ids) > 1:
        model = nn.DataParallel(model, device_ids=args.device_ids)


    if args.weights==0.5:
        criterion = nn.CrossEntropyLoss().to(device)
    else:
        w = torch.Tensor([1-args.weights,args.weights])
        criterion = nn.CrossEntropyLoss(w).to(device)
    optimizer = optim.Adam(model.parameters(), lr=args.lr, weight_decay=1e-4)
    scheduler = MultiStepLR(optimizer, milestones=[10,20], gamma=0.1, last_epoch=-1)
    cudnn.benchmark = True

    # normalization
    normalize = transforms.Normalize(mean=[0.5,0.5,0.5],std=[0.1,0.1,0.1])
    trans = transforms.Compose([
        transforms.ToTensor(),
        normalize])

    # load data
    inference_dset = Inferencedataset(args.train_lib, trans)
    inference_loader = torch.utils.data.DataLoader(
        inference_dset,
        batch_size=args.batch_size, shuffle=False,
        num_workers=args.workers, pin_memory=False)
    if args.val_lib:
        val_dset = Inferencedataset(args.val_lib, trans)
        val_loader = torch.utils.data.DataLoader(
            val_dset,
            batch_size=args.batch_size, shuffle=False,
            num_workers=args.workers, pin_memory=False)
    test_dset = Inferencedataset(args.test_lib, trans)
    test_loader = torch.utils.data.DataLoader(
        test_dset,
        batch_size=args.batch_size, shuffle=False,
        num_workers=args.workers, pin_memory=False)


    columns = []
    for i in range(1, 513):
        columns.append('feature' + str(i))

    epoch = 0
    # initialize the model saving path
    model_save_path = args.mil_model
    early_stopping = EarlyStopping(model_path=model_save_path,
                                   patience=3, verbose=True)
    if not args.is_test:
        #loop throuh epochs
        for epoch in range(args.nepochs):
            begin = time.time()
            
            if epoch == args.nepochs-1 and args.save_feat:
                probs, features = inference(epoch, inference_loader, model, args.save_feat)
            else:
                probs, _ = inference(epoch, inference_loader, model)

            topk = group_argtopk(np.array(inference_dset.slideIDX), probs, args.k)
            t_data = inference_dset.maketraindata(topk)
            train_dset = Traindataset(t_data, args.train_lib, trans)
            train_loader = torch.utils.data.DataLoader(
                 train_dset,
                 batch_size=128, shuffle=True,
                 num_workers=args.workers, pin_memory=False)

            if epoch == args.nepochs-1 and args.save_index:
                topn = group_argtopk(np.array(inference_dset.slideIDX), probs, args.n)
                inference_dset.savetopndata(topn, filename=f"select_train")
                if args.save_feat:
                    slideIDX = np.array(inference_dset.slideIDX)
                    for i, slide in enumerate(inference_dset.slidenames):
                        slidename = os.path.basename(slide).split('.')[0]
                        feature = features[slideIDX == i, :]

                        # save features
                        if args.feat_format == '.csv':
                            df = pd.DataFrame(feature, columns=columns)
                            df.to_csv(os.path.join(args.feat_dir, f'{slidename}.csv'))
                        elif args.feat_format == '.npy':
                            np.save(os.path.join(args.feat_dir, f'{slidename}.npy'), feature)
                        elif args.feat_format == '.pt':
                            feature = torch.from_numpy(feature)
                            torch.save(feature, os.path.join(args.feat_dir, f'{slidename}.pt'))

            train_dset.shuffletraindata()
            loss = train(epoch, train_loader, model, criterion, optimizer)
            end = time.time()
            usetime = end-begin
            print('Training\tEpoch: [{}/{}]\tLoss: {}\tUsetime: {:.4f}'.format(epoch+1, args.nepochs, loss, usetime))
            scheduler.step()


            #Validation
            if args.val_lib and (epoch+1) % args.test_every == 0:
                if epoch == args.nepochs-1 and args.save_feat:
                    probs, features = inference(epoch, val_loader, model, args.save_feat)
                else:
                    probs, _ = inference(epoch, val_loader, model)
                topk = group_argtopk(np.array(val_dset.slideIDX), probs, args.k)
                maxs = group_max(np.array(val_dset.slideIDX), probs, len(val_dset.targets))
                t_data = val_dset.maketraindata(topk)
                train_dset = Traindataset(t_data, args.val_lib, trans)
                train_loader = torch.utils.data.DataLoader(
                    train_dset,
                    batch_size=128, shuffle=True,
                    num_workers=args.workers, pin_memory=False)
                val_loss = train(epoch, train_loader, model, criterion, optimizer)
                if epoch == args.nepochs-1 and args.save_index:
                    topn = group_argtopk(np.array(val_dset.slideIDX), probs, args.n)
                    val_dset.savetopndata(topn, filename=f"select_val")
                    if args.save_feat:
                        slideIDX = np.array(val_dset.slideIDX)
                        for i, slide in enumerate(val_dset.slidenames):
                            slidename = os.path.basename(slide).split('.')[0]
                            feature = features[slideIDX == i, :]
                            
                            # save features
                            if args.feat_format == '.csv':
                                df = pd.DataFrame(feature, columns=columns)
                                df.to_csv(os.path.join(args.feat_dir, f'{slidename}.csv'))
                            elif args.feat_format == '.npy':
                                np.save(os.path.join(args.feat_dir, f'{slidename}.npy'), feature)
                            elif args.feat_format == '.pt':
                                feature = torch.from_numpy(feature)
                                torch.save(feature, os.path.join(args.feat_dir, f'{slidename}.pt'))
                
                pred = [1 if x >= 0.5 else 0 for x in maxs]
                fpr, tpr, thresh = roc_curve(y_true=val_dset.targets, y_score=maxs, pos_label=1)
                roc_auc = auc(fpr, tpr)
                err, fpr, fnr = calc_err(pred, val_dset.targets)

                print("Validation Accuracy: {:.4f}\t AUC: {:.4f}".format(1 - err, roc_auc))
                print('Validation\tEpoch: [{}/{}]\tError: {}\tFPR: {}\tFNR: {}'.format(epoch+1, args.nepochs, err, fpr, fnr))

                if 1 - err >= best_acc:
                    best_acc = 1 - err

            if args.test_lib and (epoch+1) % args.test_every == 0:
                ## Test
                if epoch == args.nepochs-1 and args.save_feat:
                    probs, features = inference(epoch, test_loader, model, args.save_feat)
                else:
                    probs, _ = inference(epoch, test_loader, model)
                maxs = group_max(np.array(test_dset.slideIDX), probs, len(test_dset.targets))
                if epoch == args.nepochs-1 and args.save_index:
                    topn = group_argtopk(np.array(test_dset.slideIDX), probs, args.n)
                    test_dset.savetopndata(topn, filename=f"select_test")
                    if args.save_feat:
                        slideIDX = np.array(test_dset.slideIDX)
                        for i, slide in enumerate(val_dset.slidenames):
                            slidename = os.path.basename(slide).split('.')[0]
                            feature = features[slideIDX == i, :]
                            
                            # save features
                            if args.feat_format == '.csv':
                                df = pd.DataFrame(feature, columns=columns)
                                df.to_csv(os.path.join(args.feat_dir, f'{slidename}.csv'))
                            elif args.feat_format == '.npy':
                                np.save(os.path.join(args.feat_dir, f'{slidename}.npy'), feature)
                            elif args.feat_format == '.pt':
                                feature = torch.from_numpy(feature)
                                torch.save(feature, os.path.join(args.feat_dir, f'{slidename}.pt'))
                
                pred = [1 if x >= 0.5 else 0 for x in maxs]
                fpr, tpr, thresh = roc_curve(y_true=test_dset.targets, y_score=maxs, pos_label=1)
                roc_auc = auc(fpr, tpr)
                err, fpr, fnr = calc_err(pred, test_dset.targets)
                print("Test Accuracy: {:.4f}\t AUC: {:.4f}".format(1 - err, roc_auc))
                print('Test\tError: {}\tFPR: {}\tFNR: {}'.format(err, fpr, fnr))

                ## early stop
                early_stopping(val_loss, best_acc, model)
                if early_stopping.early_stop:
                    print("Early stopping")
                    break

                ## Save the model
                if args.save_model:
                    underlying_model = model.module if hasattr(model, 'module') else model
                    obj = {
                        'state_dict': underlying_model.state_dict(),
                        'best_acc': best_acc,
                    }
                    torch.save(obj, model_save_path)

    ## Test
    if os.path.exists(model_save_path):
        ch = torch.load(model_save_path, map_location='cpu', weights_only=False)
        print(ch['best_acc'])
        model.load_state_dict(ch["state_dict"], strict=False)
    else:
        print(f"Note: no checkpoint was saved at {model_save_path} "
              f"(this happens if --nepochs is smaller than --test_every, "
              f"so validation/testing/saving never triggered). "
              f"Using the current in-memory model from the final epoch instead.")
    probs, _ = inference(epoch, test_loader, model)
    maxs = group_max(np.array(test_dset.slideIDX), probs, len(test_dset.targets))
    if args.save_index:
        topn = group_argtopk(np.array(test_dset.slideIDX), probs, args.n)
        test_dset.savetopndata(topn, filename=f"select_test")
    pred = [1 if x >= 0.5 else 0 for x in maxs]
    fpr, tpr, thresh = roc_curve(y_true=test_dset.targets, y_score=maxs, pos_label=1)
    roc_auc = auc(fpr, tpr)
    err, fpr, fnr = calc_err(pred, test_dset.targets)
    print("Test Accuracy: {:.4f}\t AUC: {:.4f}".format(1 - err, roc_auc))
    miss_wsi(test_dset, pred)



def miss_wsi(dset, pred):
    slidenames = dset.slidenames
    wsi_names = [os.path.basename(slidename).split('.')[0] for slidename in slidenames]
    wsi_names = np.array(wsi_names)
    targets = np.array(dset.targets)
    pred = np.array(pred)
    miss_wsi_name = wsi_names[pred!=targets]
    print("Misclassify {} wsi".format(miss_wsi_name.shape[0]))
    print(miss_wsi_name)


def inference(run, loader, model, save_feat=False):
    model.eval()
    probs = torch.FloatTensor(len(loader.dataset))
    prefetcher = data_prefetcher(loader)
    features = torch.Tensor()
    with torch.no_grad():
        input = prefetcher.next()
        i = 0
        while input is not None:
            if i % 1000 == 999:
                print('Inference\tEpoch: [{}/{}]\tBatch: [{}/{}]'.format(run+1, args.nepochs, i+1, len(loader)))

            input = input.to(device)
            output, feature = model(input)
            output = F.softmax(output, dim=1)
            probs[i*args.batch_size:i*args.batch_size+input.size(0)] = output.detach()[:,1].clone()
            input = prefetcher.next()
            i += 1
            if save_feat:
                features = torch.cat((features, feature.cpu()), dim=0)
    return probs.cpu().numpy(), features.cpu().numpy()


def train(run, loader, model, criterion, optimizer):
    model.train()
    running_loss = 0.
    for i, (input, target) in enumerate(loader):
        input = input.to(device)
        target = target.to(device)
        output, _ = model(input)
        loss = criterion(output, target)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        running_loss += loss.item()*input.size(0)
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    return running_loss/len(loader.dataset)


def pred(run, loader, model, criterion, optimizer):
    model.eval()
    running_loss = 0.
    with torch.no_grad():
        for i, (input, target) in enumerate(loader):
            input = input.to(device)
            target = target.to(device)
            output, _ = model(input)
            loss = criterion(output, target)
            running_loss += loss.item()*input.size(0)
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
    return running_loss/len(loader.dataset)


def calc_err(pred,real):
    pred = np.array(pred)
    real = np.array(real)
    neq = np.not_equal(pred, real)
    err = float(neq.sum())/pred.shape[0]
    fpr = float(np.logical_and(pred==1,neq).sum())/(real==0).sum()
    fnr = float(np.logical_and(pred==0,neq).sum())/(real==1).sum()
    return err, fpr, fnr

def optimal_thresh(fpr, tpr, thresholds, p=0):
    loss = (fpr - tpr) - p * tpr / (fpr + tpr + 1)
    idx = np.argmin(loss, axis=0)
    return fpr[idx], tpr[idx], thresholds[idx]

def group_argtopk(groups, data,k=1):
    order = np.lexsort((data, groups))
    groups = groups[order]
    data = data[order]
    index = np.empty(len(groups), 'bool')
    index[-k:] = True
    index[:-k] = groups[k:] != groups[:-k]
    return list(order[index])

def group_max(groups, data, nmax):
    out = np.empty(nmax)
    out[:] = np.nan
    order = np.lexsort((data, groups))
    groups = groups[order]
    data = data[order]
    index = np.empty(len(groups), 'bool')
    index[-1] = True
    index[:-1] = groups[1:] != groups[:-1]
    out[groups[index]] = data[index]
    return out



class data_prefetcher():
    def __init__(self, loader):
        self.loader = iter(loader)
        self.stream = torch.cuda.Stream() if torch.cuda.is_available() else None
        self.preload()

    def preload(self):
        try:
            self.next_input = next(self.loader)
        except StopIteration:
            self.next_input = None
            return
        if self.stream is not None:
            with torch.cuda.stream(self.stream):
                self.next_input = self.next_input.to(device, non_blocking=True)
                self.next_input = self.next_input.float()
        else:
            self.next_input = self.next_input.to(device).float()

    def next(self):
        if self.stream is not None:
            torch.cuda.current_stream().wait_stream(self.stream)
        input = self.next_input
        self.preload()
        return input



class Inferencedataset(data.Dataset):
    def __init__(self, libraryfile='', transform=None):
        lib = torch.load(libraryfile, map_location='cpu', weights_only=False)
        # Patches are read on-demand straight from each source WSI (see
        # slide_cache.py) using the exact coordinate/level already recorded
        # in this lib file, instead of requiring every slide to be
        # pre-tiled to .jpg on disk beforehand.
        slide_paths = lib['slides']
        print('')
        #Flatten grid
        grid = []
        slideIDX = []
        for i,g in enumerate(lib['grid']):
            grid.extend(g)
            slideIDX.extend([i]*len(g))

        print('Number of tiles: {}'.format(len(grid)))
        self.slidenames = lib['slides']
        self.slide_paths = slide_paths
        self.targets = lib['targets']
        self.grid = grid
        self.slideIDX = slideIDX
        self.transform = transform
        self.level = lib.get('level', 0)
        self.mult = lib.get('mult', 1)
        self.size = 256

    def savetopndata(self,idxs, filename):
        slides = []
        grids = []
        targets = []
        topngrid = [self.grid[x] for x in idxs]
        topnid = [self.slideIDX[x] for x in idxs]
        topngrid = np.array(topngrid)
        topnid = np.array(topnid)
        for i in range(len(self.slidenames)):
            slides.append(self.slidenames[i])
            grid = topngrid[topnid==i]
            grid = grid.tolist()
            grids.append(grid)
            targets.append(self.targets[i])
        torch.save({
            'slides': slides,
            'grid': grids,
            'gridIDX': list(topnid),
            'targets': targets},
            os.path.join(args.output, f'{filename}.ckpt'))
            
    def maketraindata(self, idxs):
        self.t_data = [(self.slideIDX[x],self.grid[x],self.targets[self.slideIDX[x]]) for x in idxs]
        return self.t_data
        
    def __getitem__(self,index):
        slideIDX = self.slideIDX[index]
        coord = self.grid[index]
        slide_path = self.slide_paths[slideIDX]
        img = slide_cache.get_patch(slide_path, coord, level=self.level, size=self.size, mult=self.mult)
        if self.transform is not None:
            img = self.transform(img)
        return img
    def __len__(self):
        return len(self.grid)



class Traindataset(data.Dataset):
    def __init__(self, data, libraryfile='', transform=None):
        lib = torch.load(libraryfile, map_location='cpu', weights_only=False)
        self.t_data = data
        self.slide_paths = lib['slides']
        self.level = lib.get('level', 0)
        self.mult = lib.get('mult', 1)
        self.size = 256
        self.transform = transform

    def shuffletraindata(self):
        self.t_data = random.sample(self.t_data, len(self.t_data))
    def __getitem__(self,index):
        slideIDX, coord, target = self.t_data[index]
        slide_path = self.slide_paths[slideIDX]
        img = slide_cache.get_patch(slide_path, coord, level=self.level, size=self.size, mult=self.mult)
        if self.transform is not None:
            img = self.transform(img)
        return img, target
    def __len__(self):
        return len(self.t_data)

class ResNetEncoder(nn.Module):
    def __init__(self):
        super(ResNetEncoder, self).__init__()
        temp = resnet34(pretrained=True)
        temp.fc = nn.Linear(temp.fc.in_features, 2)
        self.feature_extractor = nn.Sequential(*list(temp.children())[:-1])
        self.fc = temp.fc

    def forward(self, x):
        x = self.feature_extractor(x)
        x = x.view(x.size(0), -1)
        return self.fc(x), x


if __name__ == '__main__':
    main()


### `pmil.py` — prototype-based slide classifier training (lazy feature loading, GPU auto-detect)

In [ ]:
%%writefile pmil.py
import numpy as np
import torch.backends.cudnn as cudnn
import torch.optim as optim
from torch.optim.lr_scheduler import MultiStepLR
import os
from sklearn.metrics import roc_curve, auc
import pandas as pd
import argparse
import torch.utils.data as data
import torch.nn as nn
import torch.nn.functional as F
import torch
from model.model_pmil import IClassifier, BClassifier, PBMIL, Euclidean_Similarity


np.random.seed(24)
torch.manual_seed(24)
torch.cuda.manual_seed(24)
torch.cuda.manual_seed_all(24)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

parser = argparse.ArgumentParser(description='Generating slide embedding features through prototypes and classification')
parser.add_argument('--train_lib', type=str, default='lib/train.ckpt',
                    help='lib to save wsi id of train set')
parser.add_argument('--val_lib', type=str, default='lib/val.ckpt',
                    help='lib to save wsi id of valid set')
parser.add_argument('--test_lib', type=str, default='lib/test.ckpt',
                    help='lib to save wsi id of test set')
parser.add_argument('--feat_dir', type=str, default='feat', help='path to save features')
parser.add_argument('--output', type=str, default='result', help='output directory')
parser.add_argument('--num_cluster', type=int, help='number of cluster')
parser.add_argument('--batch_size', type=int, default=1, help='mini-batch size')
parser.add_argument('--workers', default=1, type=int, help='number of data loading workers')
parser.add_argument('--nepochs', type=int, default=30, help='number of epochs')
parser.add_argument('--lr', type=float, default=1e-4, help='learning rate')
parser.add_argument('--weights', default=0.5, type=float,
                    help='unbalanced positive class weight (default: 0.5, balanced classes)')
parser.add_argument('--global_cluster', type=str, default='cluster/prototypes_features_40x256.npy')
parser.add_argument('--pmil_model', type=str, default='model/pmil_model.pth', help='path to pretrained model')
parser.add_argument('--mil_model', type=str, default='model/mil.pth')
parser.add_argument('--s', default=5, type=int, help='how many top k patchess to consider')
parser.add_argument('--save_model', default=False, action='store_true')
parser.add_argument('--load_model', default=False, action='store_true')
parser.add_argument('--is_test', default=False, action='store_true')
parser.add_argument('--device', type=int, default=0)
parser.add_argument('--feat_format', type=str, choices = ['.csv', '.npy', '.pt'], default='.csv')

global args
args = parser.parse_args()
if torch.cuda.is_available():
    torch.cuda.set_device(args.device)
    device = torch.device(f'cuda:{args.device}')
else:
    device = torch.device('cpu')


def main():
    best_acc = 0
    pmil_model_dir = os.path.dirname(args.pmil_model)
    if pmil_model_dir:
        os.makedirs(pmil_model_dir, exist_ok=True)

    ## Loading prototype feature
    if os.path.basename(args.global_cluster).split('.')[-1] == 'npy':
        data = np.load(args.global_cluster)
        prototypes_features = data
    elif os.path.basename(args.global_cluster).split('.')[-1] == 'npz':
        data = np.load(args.global_cluster)
        prototypes_features = data['feature']
    
    args.num_cluster = prototypes_features.shape[0]
    print(f'There are {args.num_cluster} prototypes!')
    
    prototypes_features = torch.from_numpy(prototypes_features)
    prototypes_features = prototypes_features.to(device)

    ## Creating model
    i_classifier = IClassifier(mil_model=args.mil_model, feature_size=512, out_class=2).to(device)
    b_classifier = BClassifier(num_cluster=args.num_cluster, feature_size=512, input_size=128, output_class=2, dropout=0).to(device)
    model = PBMIL(i_classifier, b_classifier).to(device)
    if args.load_model:
        ch = torch.load(args.pmil_model, map_location='cpu', weights_only=False)
        model.load_state_dict(ch['state_dict'])
    if args.weights == 0.5:
        criterion = nn.CrossEntropyLoss().to(device)
    else:
        w = torch.Tensor([1 - args.weights, args.weights])
        criterion = nn.CrossEntropyLoss(weight=w).to(device)

    ## Define optimizer
    optimizer = optim.Adam(model.parameters(), lr=args.lr, weight_decay=1e-4)
    scheduler = MultiStepLR(optimizer, milestones=[10, 20], gamma=0.1, last_epoch=-1)
    cudnn.benchmark = True

    # Loading dataset
    if args.train_lib:
        train_dset = GlobalDataset(feat_dir=args.feat_dir, libraryfile=args.train_lib, is_train=True)
        train_loader = torch.utils.data.DataLoader(
            train_dset,
            batch_size=args.batch_size, shuffle=True,
            num_workers=args.workers, pin_memory=False)
    if args.val_lib:
        val_dset = GlobalDataset(feat_dir=args.feat_dir, libraryfile=args.val_lib, is_train=False)
        val_loader = torch.utils.data.DataLoader(
            val_dset,
            batch_size=args.batch_size, shuffle=False,
            num_workers=args.workers, pin_memory=False)
    test_dset = GlobalDataset(feat_dir=args.feat_dir, libraryfile=args.test_lib, is_train=False)
    test_loader = torch.utils.data.DataLoader(
        test_dset,
        batch_size=args.batch_size, shuffle=False,
        num_workers=args.workers, pin_memory=False)

    prototype = prototypes_features
    # loop throuh epochs
    if not args.is_test:
        for epoch in range(args.nepochs):
            ## train
            prototype = prototypes_features
            loss, acc = train(epoch, prototype, train_loader, model, criterion, optimizer)
            scheduler.step()

            ## Validation
            val_loss, probs, val_acc, val_auc, idx, codings = test(epoch, prototype, val_loader, model, criterion)
            targets = np.array(test_dset.targets)
            print('Testing Loss: {:.4f}\tAcc: {:.4f}\tAUC: {:.4f}'.format(val_loss, val_acc, val_auc))

            ## Save model
            if val_acc >= best_acc:
                best_acc = val_acc
                best_auc = val_auc
                best_loss = val_loss
                print("Best acc: {:.4f} | Best auc: {:.4f}".format(best_acc, best_auc))

                if args.save_model:
                    obj = {
                        'epoch': epoch + 1,
                        'i_state_dict': model.i_classifier.state_dict(),
                        'state_dict': model.state_dict(),
                        'best_acc': best_acc,
                        'best_auc': best_auc,
                        'idx': idx,
                        'features': codings,
                        'targets': targets
                    }
                    torch.save(obj, args.pmil_model)

    ## Testing
    ch = torch.load(args.pmil_model, map_location='cpu', weights_only=False)
    model.load_state_dict(ch['state_dict'])
    test_loss, probs, test_acc, auc, idx, codings = test(0, prototype, test_loader, model, criterion)
    print('Testing Loss: {:.4f}\tAcc: {:.4f}\tAUC: {:.4f}'.format(test_loss, test_acc, auc))


def train(run, prototype, loader, model, criterion, optimizer):
    model.eval()
    running_loss = 0.
    running_bag_loss = 0.
    running_max_loss = 0.
    probs = []
    targets = []
    for i, (input, target, patient_id) in enumerate(loader):
        input = input[0]
        input = input.to(device)
        target = target.to(device)

        ins_prediction, bag_prediction, _ = model(input, prototype)

        score = F.softmax(ins_prediction, dim=1)
        _, max_id = torch.max(score[:, 1], 0)

        max_prediction = ins_prediction[max_id, :]
        bag_loss = criterion(bag_prediction.view(1, -1), target)
        max_loss = criterion(max_prediction.view(1, -1), target)
        loss = bag_loss + max_loss

        running_bag_loss += bag_loss.item()
        running_max_loss += max_loss.item()

        train_prediction = F.softmax(bag_prediction.detach(),  dim=1).squeeze().cpu().numpy()
        probs.append(train_prediction[1])
        targets.append(target.item())
        optimizer.zero_grad()
        loss.backward(retain_graph=True)
        optimizer.step()
        running_loss += loss.item()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    targets = np.array(targets)
    fpr, tpr, thresholds = roc_curve(y_true=targets, y_score=probs, pos_label=1)
    roc_auc = auc(fpr, tpr)
    preds = [1 if x >= 0.5 else 0 for x in probs]
    preds = np.array(preds)
    eq = np.equal(targets, preds)
    acc = float(eq.sum()) / targets.shape[0]

    running_loss = running_loss / len(loader.dataset)
    running_max_loss = running_max_loss / len(loader.dataset)
    running_bag_loss = running_bag_loss / len(loader.dataset)
    print('Training\tEpoch: [{}/{}]\tMax Loss: {:.4f}\tBag Loss: {:.4f}\tAcc: {:.4f}AUC: {:.4f}'.format(run + 1,
                                                                                                        args.nepochs,
                                                                                                        running_max_loss,
                                                                                                        running_bag_loss,
                                                                                                        acc, roc_auc))
    return running_loss, acc


def test(run, prototype, loader, model, criterion):
    model.eval()
    running_loss = 0.
    running_bag_loss = 0.
    running_max_loss = 0.
    probs = []
    targets = []
    ids = []
    codings = []
    with torch.no_grad():
        for i, (input, target, patient_id) in enumerate(loader):
            input = input[0]
            input = input.to(device)
            target = target.to(device)
            ins_prediction, bag_prediction, coding = model(input, prototype)
            codings.append(coding.cpu().numpy())
            score = F.softmax(ins_prediction, dim=1)
            _, max_id = torch.max(score[:, 1], 0)
            _, id = torch.topk(score[:, 1], dim=0, k=5)

            ids.append(id)
            max_prediction = ins_prediction[max_id, :]
            bag_loss = criterion(bag_prediction.view(1, -1), target)
            max_loss = criterion(max_prediction.view(1, -1), target)
            loss = bag_loss + max_loss
            running_bag_loss += bag_loss.item()
            running_max_loss += max_loss.item()
            test_prediction = F.softmax(bag_prediction.detach(), dim=1).squeeze().cpu().numpy()
            probs.append(test_prediction[1])
            targets.append(target.item())

            running_loss += loss.item()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

    codings = np.array(codings)
    targets = np.array(targets)
    fpr, tpr, thresholds = roc_curve(y_true=targets, y_score=probs, pos_label=1)
    roc_auc = auc(fpr, tpr)
    preds = [1 if x >= 0.5 else 0 for x in probs]
    preds = np.array(preds)
    eq = np.equal(targets, preds)
    acc = float(eq.sum()) / targets.shape[0]
    running_loss = running_loss / len(loader.dataset)
    return running_loss, np.array(probs), acc, roc_auc, ids, codings


def optimal_thresh(fpr, tpr, thresholds, p=0):
    loss = (fpr - tpr) - p * tpr / (fpr + tpr + 1)
    idx = np.argmin(loss, axis=0)
    return fpr[idx], tpr[idx], thresholds[idx]



class GlobalDataset(data.Dataset):
    columns = []
    for i in range(1, 513):
        columns.append('feature' + str(i))

    def __init__(self, feat_dir='', libraryfile='', is_train=True):
        self.feat_dir = feat_dir
        self.is_train = is_train

        lib = torch.load(libraryfile, weights_only=False)
        self.slidenames = [os.path.basename(slide).split('.')[0] for slide in lib['slides']]
        gt_targets = lib['targets']
        self.targets = self._get_label(gt_targets)
        # Feature files are read one slide at a time in __getitem__ instead
        # of all being loaded into RAM here up front -- with batch_size=1
        # this means only one slide's features are resident at any moment,
        # regardless of how many slides are in the dataset.

    def _get_label(self, gt_targets):
        targets = []
        for slidename in self.slidenames:
            idx = self.slidenames.index(slidename)
            targets.append(gt_targets[idx])
        return targets

    def _load_slide_feat(self, slidename):
        if args.feat_format == '.csv':
            df = pd.read_csv(os.path.join(self.feat_dir, slidename + '.csv'))
            feat = df[self.columns].values
        elif args.feat_format == '.npy':
            feat = np.load(os.path.join(self.feat_dir, slidename + '.npy'))
        elif args.feat_format == '.pt':
            feat = torch.load(os.path.join(self.feat_dir, slidename + '.pt'), weights_only=False)
            feat = feat.numpy()
        return feat.astype(np.float32)

    def __getitem__(self, idx):
        slidename = self.slidenames[idx]
        feat = self._load_slide_feat(slidename)
        target = self.targets[idx]
        patient_id = slidename
        return feat, target, patient_id

    def __len__(self):
        return len(self.slidenames)


if __name__ == '__main__':
    main()


### `cluster.py` — local + global prototype clustering (CPU-only, unmodified from upstream)

In [ ]:
%%writefile cluster.py
from sklearn.cluster import KMeans, AffinityPropagation
from sklearn.metrics.pairwise import pairwise_distances, paired_cosine_distances
import pandas as pd
import numpy as np
import os
import argparse
import time
import torch
import pickle


parser = argparse.ArgumentParser(description='Using AP clustering to discover the prototypes')
parser.add_argument('--lib', type=str, default='lib/select_train.ckpt', help='lib to save wsi id of train set')
parser.add_argument('--preference', type=int, default=0, help='preference for AP clustering')
parser.add_argument('--damping', type=int, default=0.5, help='damping for AP clustering')
parser.add_argument('--global_cluster', type=str, default='cluster/prototypes_feature.npy')
parser.add_argument('--prototypes_index', type=str, default='cluster/prototypes_index.ckpt')
parser.add_argument('--global_ap_model', type=str, default='cluster/ap_model.pkl')
parser.add_argument('--feat_dir', type=str, default='', help='path to save features')
parser.add_argument('--local_cluster', type=str, default='cluster/local_cluster.ckpt')
parser.add_argument('--lamb', type=float, default=0.25)
parser.add_argument('--feat_format', type=str, choices = ['.csv', '.npy', '.pt'], default='.csv')

global args
args = parser.parse_args()

def main():
    for path in [args.global_cluster, args.prototypes_index, args.global_ap_model, args.local_cluster]:
        d = os.path.dirname(path)
        if d:
            os.makedirs(d, exist_ok=True)

    feature_columns = make_csv_columns()
    feat_paths = get_feature_path()
    
    # intra-slide clustering (WSI-level)
    local_cluster_centroids, local_centroids_features, slideIDXs, gridIDXs = local_clustering(feat_paths, feature_columns)
    
    # inter-slide clustering (whole dataset)
    global_cluster_centroids, global_centroids_features, ap = global_clustering(local_centroids_features)

    # Saving AP clustering model
    with open(args.global_ap_model, 'wb') as f:
        pickle.dump(ap, f)

    # Saving the global prototype features
    np.savez(args.global_cluster, feature=global_centroids_features)

    # save global cluster centroids index
    save_global_centroids_idx(global_cluster_centroids, slideIDXs, gridIDXs)

def local_clustering(feature_paths, feature_columns):
    local_cluster_centroids = []
    wsi_names = []
    slideIDXs = []
    cluster_gridIDXs = []
    local_centroids_features = torch.Tensor()

    for i, file in enumerate(feature_paths):
        wsi_name = file.split('.')[0]
        wsi_names.append(wsi_name)
        
        if args.feat_format == '.csv':
            df = pd.read_csv(os.path.join(args.feat_dir, file), index_col=0)
            feat = df[feature_columns].values
        elif args.feat_format == '.npy':
            feat = np.load(os.path.join(args.feat_dir, file))
        elif args.feat_format == '.pt':
            feat = torch.load(os.path.join(args.feat_dir, file), weights_only=False)
        
        begin = time.time()
        similarity = euclidean_similarity(feat)
   
        # default using negative squared euclidean distance
        ap = AffinityPropagation(preference=args.preference, damping=args.damping, affinity='precomputed', random_state=24).fit(similarity)
        cluster_centers_indices = ap.cluster_centers_indices_
        n_clusters = len(cluster_centers_indices)
        end = time.time()
        local_cluster_centroids.append(cluster_centers_indices)
        usetime = end - begin
        print("wsi {}\t File name: {}\t Use time: {:.2f}\t Number of cluster: {}".format(i + 1, file, usetime,n_clusters))
        slideIDXs.extend([i] * n_clusters)
        cluster_gridIDXs.extend(cluster_centers_indices)
        local_centroids_feature = feat[cluster_centers_indices, :]
        local_centroids_feature = local_centroids_feature.astype(np.float32)
        local_centroids_feature = torch.from_numpy(local_centroids_feature)
        local_centroids_features = torch.cat((local_centroids_features, local_centroids_feature), dim=0)
    torch.save({
        'preference': args.preference,
        'dampling': args.damping,
        'wsi_names': wsi_names,
        'centroid': local_cluster_centroids},
        args.local_cluster)
    return local_cluster_centroids, local_centroids_features, slideIDXs, cluster_gridIDXs

def global_clustering(local_centroids_features):
    local_centroids_features = np.array(local_centroids_features)
    similarity = euclidean_similarity(local_centroids_features)
    ap = AffinityPropagation(preference=args.preference, damping=args.damping, affinity='precomputed', random_state=24).fit(similarity)
    cluster_centers_indices = ap.cluster_centers_indices_
    labels = ap.labels_
    n_clusters = len(cluster_centers_indices)
    print("Estimate number of cluster: ", n_clusters)
    global_cluster_centroids = cluster_centers_indices
    global_centroids_features = local_centroids_features[global_cluster_centroids, :]
    global_centroids_features = global_centroids_features.astype(np.float32)
    return global_cluster_centroids, global_centroids_features, ap

def save_global_centroids_idx(global_cluster_centroids, slideIDXs, gridIDXs):
    centroids_slideIDXs = []
    centroids_idx = []
    for idx in global_cluster_centroids:
        slideIDX = slideIDXs[idx]
        centroids_slideIDXs.append(slideIDX)
        centroids_idx.append(gridIDXs[idx])
    torch.save({
        'slideIDX': centroids_slideIDXs,
        'gridIDX': centroids_idx},
        args.prototypes_index)

def cosine_distance(matrix1,matrix2):
    matrix1_matrix2 = np.dot(matrix1, matrix2.transpose())
    matrix1_norm = np.sqrt(np.multiply(matrix1, matrix1).sum(axis=1))
    matrix1_norm = matrix1_norm[:, np.newaxis]
    matrix2_norm = np.sqrt(np.multiply(matrix2, matrix2).sum(axis=1))
    matrix2_norm = matrix2_norm[:, np.newaxis]
    cosine_distance = np.divide(matrix1_matrix2, np.dot(matrix1_norm, matrix2_norm.transpose()))
    return cosine_distance
    
def euclidean_similarity(x):
    dists = pairwise_distances(x, metric='euclidean')
    dists =  dists / np.max(dists)
    similarity = np.exp(-dists * args.lamb)
    return similarity

def make_csv_columns(ndim=512):
    feature_columns = []
    for i in range(1, ndim+1):
        feature_columns.append('feature' + str(i))
    return feature_columns

def get_feature_path():
    lib = torch.load(args.lib, map_location='cpu', weights_only=False)
    wsi_paths = lib['slides']
    wsi_names = [os.path.basename(wsi_path).split('.')[0] for wsi_path in wsi_paths]
    feat_paths = [wsi_name + args.feat_format for wsi_name in wsi_names]
    return feat_paths
    
def get_grid_index():
    gridIDXs = {}
    lib = torch.load(args.lib, map_location='cpu', weights_only=False)
    wsi_paths = lib['slides']
    index = lib['gridIDX']
    wsi_names = [os.path.basename(wsi_path).split('.')[0] for wsi_path in wsi_paths]
    for i, wsi_name in enumerate(wsi_names):
        gridIDXs[wsi_name] = index[i]
    return gridIDXs

if __name__ == '__main__':
    main()


In [ ]:
!ls -la /content/PMIL /content/PMIL/model

## 3. Mount Drive and set your paths — EDIT THIS CELL

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# ---- EDIT THESE ----
PROJECT_DIR    = '/content/drive/MyDrive/PMIL'      # anything persistent on your Drive
TRAIN_LIB      = f'{PROJECT_DIR}/lib/train.ckpt'
VAL_LIB        = f'{PROJECT_DIR}/lib/val.ckpt'
TEST_LIB       = f'{PROJECT_DIR}/lib/test.ckpt'
FEAT_DIR       = f'{PROJECT_DIR}/feat'
MIL_MODEL      = f'{PROJECT_DIR}/model/mil.pth'
PMIL_MODEL     = f'{PROJECT_DIR}/model/pmil_model.pth'
GLOBAL_CLUSTER = f'{PROJECT_DIR}/cluster/global_prototypes.npy'
NEPOCHS        = 30          # how many epochs you actually intend to train max-mil.py for
# ---------------------

import os
os.makedirs(FEAT_DIR, exist_ok=True)
os.makedirs(os.path.dirname(MIL_MODEL), exist_ok=True)
os.makedirs(os.path.dirname(GLOBAL_CLUSTER), exist_ok=True)

for path, label in [(TRAIN_LIB, 'train_lib'), (VAL_LIB, 'val_lib'), (TEST_LIB, 'test_lib')]:
    print(f"{label}: {'FOUND' if os.path.exists(path) else 'MISSING -> ' + path}")


### Don't have `lib/*.ckpt` files yet?

Each one is a `torch.save()`'d dict your own code builds, with at least:

```python
{
    'slides':  [...],   # full path to each WSI, reachable from Colab (e.g. under your mounted Drive)
    'grid':    [...],   # per-slide list of (x, y) patch coordinates
    'targets': [...],   # per-slide label (0/1)
    'level':   0,       # optional, defaults to 0
    'mult':    1,       # optional, defaults to 1
}
```

Uncomment and adapt this if you need a minimal builder:

In [ ]:
# import torch
#
# lib = {
#     'slides':  ['/content/drive/MyDrive/PMIL/wsi/slide_001.tif', ...],
#     'grid':    [[(0, 0), (256, 0), ...], ...],   # one coordinate list per slide, same order as 'slides'
#     'targets': [0, 1, ...],                       # one label per slide, same order as 'slides'
#     'level': 0,
#     'mult': 1,
# }
# torch.save(lib, TRAIN_LIB)


## 4. Time estimate — smoke test before the real run

Runs `max-mil.py` for 1 epoch on just the first couple of slides in `TRAIN_LIB`, times it, then extrapolates to your real slide count and `NEPOCHS`. This only covers `max-mil.py` (the dominant cost); `cluster.py`/`pmil.py` are cheap by comparison and not included.

In [ ]:
import torch, time, subprocess, os

SMOKE_SLIDES = 2   # bump to 3-5 for a steadier estimate, at the cost of a longer smoke test

full_lib = torch.load(TRAIN_LIB, map_location='cpu')
total_slides = len(full_lib['slides'])
n_smoke = min(SMOKE_SLIDES, total_slides)

smoke_lib = {
    'slides':  full_lib['slides'][:n_smoke],
    'grid':    full_lib['grid'][:n_smoke],
    'targets': full_lib['targets'][:n_smoke],
}
if 'level' in full_lib: smoke_lib['level'] = full_lib['level']
if 'mult' in full_lib:  smoke_lib['mult']  = full_lib['mult']

smoke_lib_path = '/content/smoke_train.ckpt'
torch.save(smoke_lib, smoke_lib_path)

n_patches_smoke = sum(len(g) for g in smoke_lib['grid'])
n_patches_total = sum(len(g) for g in full_lib['grid'])

print(f"Smoke test: {n_smoke}/{total_slides} slides, {n_patches_smoke} patches "
      f"(full dataset: {n_patches_total} patches)")

start = time.time()
result = subprocess.run(
    ['python', 'max-mil.py',
     '--train_lib', smoke_lib_path,
     '--val_lib', smoke_lib_path,
     '--test_lib', smoke_lib_path,
     '--mil_model', MIL_MODEL,
     '--feat_dir', '/content/smoke_feat',
     '--nepochs', '1'],
    capture_output=True, text=True
)
elapsed = time.time() - start

os.makedirs('/content/smoke_feat', exist_ok=True)
print(result.stdout[-2000:])
if result.returncode != 0:
    print("--- STDERR (smoke run failed, fix this before the real run) ---")
    print(result.stderr[-2000:])
else:
    per_epoch_full = elapsed * (n_patches_total / n_patches_smoke)
    total_estimate_sec = per_epoch_full * NEPOCHS
    print(f"\nSmoke run (1 epoch, {n_smoke} slides): {elapsed:.1f} sec")
    print(f"Extrapolated: ~{per_epoch_full/60:.1f} min/epoch on the full dataset")
    print(f"Extrapolated: ~{total_estimate_sec/3600:.1f} hours for all {NEPOCHS} epochs "
          f"(max-mil.py only -- cluster.py/pmil.py are additional but comparatively cheap)")


## 5. Patch encoder — `max-mil.py`
Skip this cell if you already have a trained `mil.pth` at `MIL_MODEL` and just want to reuse it.

In [ ]:
!python max-mil.py \\
    --train_lib "{TRAIN_LIB}" \\
    --val_lib "{VAL_LIB}" \\
    --test_lib "{TEST_LIB}" \\
    --mil_model "{MIL_MODEL}" \\
    --feat_dir "{FEAT_DIR}" \\
    --nepochs {NEPOCHS} \\
    --save_model \\
    --save_index \\
    --save_feat


## 6. Prototype clustering — `cluster.py`
CPU-only (sklearn Affinity Propagation), needs the per-slide feature files produced above.

In [ ]:
!python cluster.py \\
    --feat_dir "{FEAT_DIR}" \\
    --lib "{PROJECT_DIR}/lib/select_train.ckpt" \\
    --save_path "{GLOBAL_CLUSTER}"


## 7. Final slide classifier — `pmil.py`

In [ ]:
!python pmil.py \\
    --train_lib "{TRAIN_LIB}" \\
    --val_lib "{VAL_LIB}" \\
    --test_lib "{TEST_LIB}" \\
    --feat_dir "{FEAT_DIR}" \\
    --mil_model "{MIL_MODEL}" \\
    --pmil_model "{PMIL_MODEL}" \\
    --global_cluster "{GLOBAL_CLUSTER}" \\
    --save_model


## Done

Trained checkpoints (`mil.pth`, `pmil_model.pth`), extracted features, and cluster prototypes are all saved under your mounted `PROJECT_DIR` on Drive, so they'll still be there next session — Colab's local `/content` disk (including the source files written in Section 2) is wiped when the runtime resets; just re-run Sections 1–2 if you start a fresh runtime.